# Reading web pages with requests and Beautiful Soup: arXiv hardware papers

**Three lessons, one notebook.** This notebook follows three lessons of the course, one after the other: part 1, [From a Web Page to Data: Beautiful Soup](https://dvirross.com/courses/intro-data-science/data-collection-beautifulsoup) (tasks 1 to 3); part 2, [From One Element to All Entries: find_all, Numbers and Missing Values](https://dvirross.com/courses/intro-data-science/data-collection-find-all) (tasks 4 to 6); part 3, [From a List to Many Pages: Following Links](https://dvirross.com/courses/intro-data-science/data-collection-linked-pages) (tasks 7 to 9). A heading marks the place where each part starts. The tasks are numbered from 1 to 9 without a break, so `check(N)` always belongs to the task N of the part that holds it.

# Part 1: From a Web Page to Data: Beautiful Soup

The lesson page of this part: [From a Web Page to Data: Beautiful Soup](https://dvirross.com/courses/intro-data-science/data-collection-beautifulsoup)

In this lesson we learn how a program asks for a web page, how the tags of an [HTML](https://dvirross.com/courses/intro-data-science/glossary#html) page are organized into elements with attributes, and how the Python library Beautiful Soup finds one element and reads its text and its attributes.

## Which hardware papers appeared this month?

[arXiv](https://arxiv.org) (pronounced "archive") is a free archive where researchers post scientific papers. Its section Hardware Architecture, `cs.AR`, lists papers on how processors, memories and accelerators are designed. The list of October 2026 is one web page with 266 entries. We want the title, the link and the number of authors of each paper. The list is shown as a web page, and a person can read the data there.

<figure class="lesson-shot">
<a href="/images/course-illustrations/data-collection/arxiv-listing-entries.webp"><img src="/images/course-illustrations/data-collection/arxiv-listing-entries.webp" width="960" height="400" alt="The beginning of the arXiv list of Hardware Architecture papers for October 2026: the line with the total number of entries, and the first entries, each with its title, its authors and links to the paper. Four numbered markers point to the total, a title, the authors and the links." loading="lazy"></a>
<figcaption>The list of October 2026, as the browser shows it. Marker 1: the number of entries in the month. Marker 2: the title of a paper. Marker 3: its authors. Marker 4: the links to its pdf, its html version and other formats.</figcaption>
</figure>

Reading data from web pages with a program is called **[web scraping](https://dvirross.com/courses/intro-data-science/glossary#web-scraping)**. A program can do what the person does if it can read the page. Counting the authors of 266 papers by eye takes an hour; a program takes a second, and it does the same for the list of the next month. In this lesson a program gets the page and reads the title and the link of the first paper from it.

## Setup: the libraries and the check function

No account and no key are needed. The next cell imports `requests` and `beautifulsoup4` (Colab has both; the cell installs them if the session does not). The pages that the code reads are saved copies of arXiv pages, kept on the public site of the course, so the numbers are the same for everyone. The cells marked **Live site** read arXiv itself, so their numbers can differ from the page, and they can fail when the site does not answer. They are switched off (`RUN_LIVE = False`), so that a whole class does not send requests to arXiv from one address at once.

In [1]:
try:
    import bs4, requests
except ImportError:
    !pip install -q requests beautifulsoup4
    import bs4, requests
print('requests', requests.__version__, '| beautifulsoup4', bs4.__version__)

requests 2.34.2 | beautifulsoup4 4.15.0


The next cell defines the function `check(task_number)`. After you run the cell of a task, run the cell under it: `check` runs your cell again, collects what it prints, and compares it with the expected output. It tells you whether the output is right, and where it differs if not; it never shows the solution.

In [2]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '1': [
        'a4ca3edcead4'
    ],
    '2': [
        '2dfe249ea3aa'
    ],
    '3': [
        'c9ad4f15cacd'
    ],
    '4': [
        'cacbc3d3f1c4'
    ],
    '5': [
        '19581e27de7c'
    ],
    '6': [
        '7b1bd53949b7'
    ],
    '7': [
        '4e07408562be'
    ],
    '8': [
        'ead42beead86'
    ],
    '9': [
        '4652614c4d87'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('There is no such task: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('The cell raised an error:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('Nothing was printed. Print the answer in the task cell, run it, and run `check` straight after it.')
    elif len(got) != len(want):
        print('The cell printed {got} line(s) and the task expects {want}. Re-read the task and the hint.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('Line {k} is different from the expected output. Re-read the task and the hint.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: the output is as expected.')


## The practice site and its saved copy

The pages of this unit are real pages of arXiv, and its **[`robots.txt`](https://dvirross.com/courses/intro-data-science/glossary#robots-txt)** (a file in which a site tells programs which of its pages they may read) allows the pages we use. The browser lets a page on this site talk only to this site, so the code on this page reads a **saved copy** of the pages, kept on this site: the list of October 2026 (its first 25 entries) and the page of each of its first eight papers. The copies are the pages as arXiv sent them on 13 November 2026, unchanged. The notebooks of this unit run the same code on the copy; their cells for the live site, where the numbers can differ, are off (`RUN_LIVE = False`): a class shares one address, and arXiv asks for 15 seconds between requests. The papers belong to their authors, the copies are kept for teaching only, and the source of every page is [arXiv.org](https://arxiv.org).

## A page is text with tags

When we type an address in the browser, the browser sends a **request** to the **server** that holds the page, and the server sends back a **response**. The exchange follows the rules of **[HTTP](https://dvirross.com/courses/intro-data-science/glossary#http)**, the way of talking of the web, and the address is a **[URL](https://dvirross.com/courses/intro-data-science/glossary#url)**. The response holds the text of the page, written in **[HTML](https://dvirross.com/courses/intro-data-science/glossary#html)**: text wrapped in **tags**, words in angle brackets such as `<dt>` and `</dt>`, which tell the browser what each piece is. The browser hides the tags and draws the page. A program that asks for the page receives the tags too.

The page is built of **elements**. An element is a piece of the page between an opening tag and a closing tag, and it can hold text and other elements. Here is the beginning of the first entry of the list, as it is in the text of the page, with parts left out:

```html
<dt><a name='item1'>[1]</a>
  <a href="/abs/2610.00106" title="Abstract">arXiv:2610.00106</a>
<dd>
  <div class='meta'>
    <div class='list-title mathjax'><span class='descriptor'>Title:</span>
      SyntheticHLS: Building Diverse Synthetic High-Level ...
    </div>
    <div class='list-authors'>
      <a href="https://arxiv.org/search/...">Stefan Abi-Karam</a>, ...
    </div>
```

- **Nesting.** The element `dd` holds the `div` of the class `meta`, which holds the `div` of the title, which holds a `span`. The inner elements are the **children** of the outer one.
- **Attributes.** An opening tag may carry attributes, pairs of a name and a value such as `href="/abs/2610.00106"` or `title="Abstract"`. The text of an element is what a reader sees; an attribute is information about the element, for example where a link goes.
- **Class.** The attribute `class` gives an element one or more names, here `list-title` and `mathjax`. Elements of the same kind share a class name, so the class is how a program tells a title from a line of authors.

The browser shows the elements of the page as a tree. A right click on a part of the page and the item **Inspect** open the developer tools, with the clicked element marked. When we want data from a page, this look decides what to write in code: which tag holds it, which class or attribute tells it from the elements around it, and whether we need its text or an attribute.

<figure class="lesson-shot">
<a href="/images/course-illustrations/data-collection/arxiv-devtools-entry.webp"><img src="/images/course-illustrations/data-collection/arxiv-devtools-entry.webp" width="773" height="462" alt="The developer tools of a browser on the arXiv list, with the elements tree open on the first entry. Four numbered markers highlight the opening tag of an element, its class attribute, its text, and the href attribute of a link." loading="lazy"></a>
<figcaption>The elements tree of the first entry. Marker 1: a tag, the name of the element. Marker 2: the attribute <code>class</code>, which tells what kind of element it is. Marker 3: the text that the element holds, the title of the paper. Marker 4: the attribute <code>href</code> of a link, the address it leads to.</figcaption>
</figure>

## Asking for the page from Python

The library `requests` sends a request from Python. The function `requests.get` sends a request that asks for a page, and returns a **response object**: everything that came back, kept in one variable. Two things in it are read first: `status_code`, a number that says how the request went (200 means that the page arrived), and `text`, the text of the page as one long string. The function `len` returns the number of characters of a string. The address is long, so it is written in two parts: two texts in quotes that stand next to each other, inside brackets, are joined by Python into one text, and the brackets let them go on in the next line.

In [3]:
import requests

list_url = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/arxiv-cs-ar-2026-10.htm'
)
response = requests.get(list_url)
print(response.status_code)
html = response.text
print(len(html))

200
46598


The page is 46,598 characters long, much more than what we see on the screen, because most of it is tags.

## Beautiful Soup reads the tags

**Beautiful Soup** is a Python library that reads the text of an HTML page and builds from it the tree of its elements, so that we can ask for an element instead of cutting text. A program that reads text in a given format and builds a structure from it is a **[parser](https://dvirross.com/courses/intro-data-science/glossary#parser)**. The library `beautifulsoup4` is imported under the name `bs4`, and the class that we use is `BeautifulSoup`. It takes two arguments: the text of the page, and the name of the parser, `'html.parser'`, the one that comes with Python.

Two more names are needed. `soup.title` is the first element of the tag `title` in the page, and the name `text` written after an element, as in `soup.title.text`, gives all the text inside it.

In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, 'html.parser')
print(soup.title)
print(soup.title.text)

<title>Hardware Architecture Oct 2026</title>
Hardware Architecture Oct 2026


The first line printed the element with its tags, and the second only the text that it holds.

## `find`: the first element that fits

The method `find` of the soup returns the first element that fits what we ask. We give it the name of a tag, and, to tell elements of the same tag apart, a class. The word `class` is a reserved word of Python, so the argument is written `class_`.

In [5]:
title = soup.find('div', class_='list-title')
print(title.text)

Title:
          SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs
        


The text of the element is what the title holds, with its inner element: the word `Title:` of the span, and the spaces and the line breaks of the page around the title. Two string methods clean it. `replace('Title:', '')` replaces a piece of text with another one, here with nothing, and `strip()` removes the spaces and the line breaks from both ends of a string.

In [6]:
print(title.text.replace('Title:', '').strip())

SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs


**Your turn.** What does `find` return when no element fits? Predict what the box prints, then run it. This page has no element of the tag `table`.

In [7]:
print(soup.find('table'))

None


**Solution**

`find` returns `None`, the value that stands for "nothing", and does not stop with an error. The error comes one step later, when the program asks for `.text` of `None`, which is not an element. A program that reads a real page has to expect that an element is not there.

## An attribute is read with `get`

The link to the page of the first paper is not text on the screen: it is the attribute `href` of an element `a`. The method `get` of an element returns the value of one attribute, given by name. To find the element, `find` accepts any attribute as an argument: the element `a` whose attribute `title` is `Abstract`.

In [8]:
link = soup.find('a', title='Abstract')
print(link.text.strip())
print(link.get('href'))

arXiv:2610.00106
/abs/2610.00106


The text of the link is what a reader sees, the identifier of the paper; its attribute `href` is where the link goes. The address starts with a slash and has no site name: it is relative to the site of the page, which is `https://arxiv.org`, so the page of this paper is `https://arxiv.org/abs/2610.00106`.

The argument `attrs` writes a condition on an attribute as a dictionary. It is needed when an attribute is named like a parameter of `find`: the first parameter is `name`, the name of the tag, so `find('a', name='item1')` raises a `TypeError`. The anchor `item1` is found with `attrs`:

In [9]:
anchor = soup.find('a', attrs={'name': 'item1'})
print(anchor.text)

[1]


## Moving inside an element

An element that `find` returned has a `find` of its own, which looks only inside it. This is how we narrow the search: first the entry, then the part of the entry. The first element `dt` of the page is the line with the identifier of the paper. Its first link, `dt.a`, is a link that has no `href`: look at what `get` returns.

In [10]:
entry = soup.find('dt')
print(entry.a)
print(entry.a.get('href'))

<a name="item1">[1]</a>
None


The first `a` in the entry is the anchor `[1]`, with the attribute `name`, which the page uses for its own jumps. It has no `href`, so `get` returns `None` and does not fail. The entry holds more links, and the link that we want is the one with `title='Abstract'`. Reading by position, "the first link", is the weak point of a program; reading by what the element is, its class or its attribute, is the strong one.

The elements inside an element can also be reached by their tag name, one level after the other, with dots: `first.div` is the first `div` inside the entry `first`, and `.div` again is the first `div` inside it, which here is the title. The method `get_text` returns the text of an element like `text` does, and takes arguments: `get_text(' ', strip=True)` joins the pieces of the text with a space and removes the spaces at the two ends.

In [11]:
first = soup.find('dd')
print(first.div.div.get_text(' ', strip=True))

Title: SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs


The path `first.div.div` depends on the place of each element, so it breaks when the page adds a `div` before the title. Reading by `class_` does not.

### Practice

**Your turn.** Print the title of the first paper of the list, without the word `Title:` and without the spaces around it, as one line.

> **Hint**
>
> The title is in the `div` of the class `list-title`. Take its `text`, remove `'Title:'` with `replace`, and clean the ends with `strip`.

**Solution**

In [12]:
title = soup.find('div', class_='list-title')
print(title.text.replace('Title:', '').strip())

SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs


**Expected output**

```text
SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs
```

In [13]:
check(1)

OK: the output is as expected.


**Common mistake**

```python
title = soup.find('div', class_='list-title')
print(title.text.strip())
```

The method `strip` removes only the spaces and the line breaks at the two ends. The word `Title:` is the text of the inner `span`, and `text` returns it with the rest, so the line starts with `Title:` and a line break.

**Your turn.** The pdf of the first paper is a link with the attribute `title` equal to `Download PDF`. Print the address in its attribute `href`.

> **Hint**
>
> As in the link to the abstract, `find('a', title='Download PDF')` returns the element, and `get('href')` returns the address.

**Solution**

In [14]:
pdf = soup.find('a', title='Download PDF')
print(pdf.get('href'))

/pdf/2610.00106


**Expected output**

```text
/pdf/2610.00106
```

In [15]:
check(2)

OK: the output is as expected.


The element is the first link on the page with that title, so it belongs to the first paper.

**Common mistake**

```python
pdf = soup.find('a', title='Download PDF')
print(pdf.text)
```

The text of the link is the word `pdf`, which a reader sees. The address is not text but the attribute `href`, so it is read with `get`.

**Your turn.** The subject of a paper is in a `span` of the class `primary-subject`. Print the text of that element for the first paper of the list.

> **Hint**
>
> `find('span', class_='primary-subject')` returns the first element of the page that fits, which is the one of the first paper.

**Solution**

In [16]:
subject = soup.find('span', class_='primary-subject')
print(subject.text)

Hardware Architecture (cs.AR)


**Expected output**

```text
Hardware Architecture (cs.AR)
```

In [17]:
check(3)

OK: the output is as expected.


The text has no spaces around it, so no `strip` is needed. The paper also lists another subject, Machine Learning, but it is text outside this element, so `find` does not return it.

**Common mistake**

```python
subject = soup.find('span', class_='subject')
print(subject)
```

There is no element with the class `subject`: the class is `primary-subject`. `find` does not fail, it returns `None`, and the program prints `None`. The class name has to be written as the page spells it, and the elements tree of the browser shows the spelling.

## Keep in mind

- An element has a tag, attributes and content, and can hold other elements. The attribute `class` tells what kind of element it is; the attribute `href` of a link is where it leads.
- `BeautifulSoup(html, 'html.parser')` builds the tree of the elements from the text of the page.
- `soup.find(tag, class_=name)` returns the first element that fits, or `None`; `find` also takes any attribute as an argument (`title='Abstract'`) or a dictionary `attrs`.
- `text` of an element is its text, `get('name')` is the value of an attribute, and `get` returns `None` when the attribute is missing.
- `replace` and `strip` clean text; an element has a `find` of its own that looks only inside it.
- Read by what the element is (class, attribute), not by its place in the page.

**Live site.** The cell below can ask arXiv itself for the first 25 entries of October 2026, with a user agent that says how to reach us. It is switched off (`RUN_LIVE = False`): a whole class that runs it together sends many requests from one address, and arXiv asks programs to wait 15 seconds between requests. One person, once, can set `RUN_LIVE = True` to see the live site. The rest of the notebook works on the saved copy.

In [18]:
RUN_LIVE = False  # switch to True to ask arXiv itself (one request)
if RUN_LIVE:
    live_headers = {'User-Agent': 'IntroDataScienceCourse/1.0 (teaching example; contact via dvirross.com)'}
    live = requests.get('https://arxiv.org/list/cs.AR/2026-10?skip=0&show=25', headers=live_headers, timeout=30)
    if live.status_code in (403, 429):
        print('The site refused the request or asked to slow down (status', live.status_code, '). Do not try again now; wait and use the saved copy.')
    else:
        print(live.status_code, len(live.text), live.text.count('<dt>'))
else:
    print('The live cell is off. Set RUN_LIVE = True to run it.')

> **Self-study, importance 1.** This part is for self-study: it is not taught in the lecture, and you read it on your own. reading pages responsibly

## Reading pages responsibly

A program that reads a site is a guest. A few rules keep it a welcome one.

- **`robots.txt`.** Every site may publish a file with this name at its top address, in which it asks programs which pages not to read and how fast to read the others. It is a request that careful programs keep, not a lock. A saved copy of the file of arXiv is kept on this site, and the box below prints the first lines of its first group: `split('User-agent:')` cuts the text at each group, and `splitlines()` returns the text of a group as a list of lines.
- **Rate.** The line `Crawl-delay: 15` asks for 15 seconds between two requests. A program that sends requests one after the other without a pause could slow the site for its readers, and would be blocked. In code the pause is one line, `time.sleep(15)`, which stops the program for 15 seconds.
- **Who is asking.** Every request carries the name of the program that sends it, the **[user agent](https://dvirross.com/courses/intro-data-science/glossary#user-agent)**. The library `requests` sends the name `python-requests` unless we change it, and some sites refuse that name: on 13 November 2026 the English Wikipedia answers it with the status code 403. A polite program sends a name of its own that says what it is and how to reach its owner, in the argument `headers` of `requests.get`.
- **Terms of use and rights.** The papers belong to their authors, and the terms of a site say what may be copied and where it may be shown. The copy of the pages on this site is for teaching, and it is not a copy for another site.
- **Personal data.** The page of a paper shows the name of the person who submitted it. The questions of these lessons do not need it, so the programs do not read it. Collect what the question needs, and think twice before building a table of people.
- **A better door.** When a site offers its data through an API, the interface made for programs, that is the polite way to ask for a lot of data. arXiv documents one for large collections.

In [19]:
robots_url = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/arxiv-robots.txt'
)
robots = requests.get(robots_url).text
first_group = robots.split('User-agent:')[1]
for line in first_group.splitlines()[:5]:
    print(line.strip())

*
Crawl-delay: 15
Allow: /archive
Allow: /year
Allow: /list


A request on the live site, with a user agent of its own and the pause between requests, looks like this:

```python
import time

headers = {'User-Agent': 'IntroDataScienceCourse/1.0 (teaching example; contact via dvirross.com)'}
for address in addresses:
    page = requests.get(address, headers=headers)
    time.sleep(15)
```

> **Self-study, importance 3.** This part is for self-study: it is not taught in the lecture, and you read it on your own. status codes and headers

## Status codes and headers

A server answers every request with a **[status code](https://dvirross.com/courses/intro-data-science/glossary#status-code)**, and a program checks it before it uses the text. A request for a page that does not exist does not fail in Python: it returns a response whose code is 404.

| Code | Meaning |
| --- | --- |
| 200 | The page arrived |
| 403 | The server refuses this request |
| 404 | There is no such page |
| 429 | Too many requests: wait |

In [20]:
missing_url = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/arxiv-no-such-page.htm'
)
missing = requests.get(missing_url)
print(missing.status_code)
print(response.headers['Content-Type'])

404
text/html; charset=utf-8


The response also carries `headers`, a dictionary of information about the answer. `Content-Type` says the kind of the content, and the text encoding that the server used.

> **Self-study, importance 4.** This part is for self-study: it is not taught in the lecture, and you read it on your own. text encoding

## Text and bytes: the encoding

The response holds the page twice: `text` is a string of characters, and `content` is the bytes that the server sent. Characters that are not in the English alphabet, such as a letter with an accent in the name of an author, take two or more bytes, so the bytes are more than the characters. The library turns the bytes into text with the encoding that the server declared in `Content-Type`.

In [21]:
print(len(response.text))
print(len(response.content))

46598
46606


When a server does not declare the encoding, the text of such a character can come out wrong. Then the bytes are decoded by hand: `response.content.decode('utf-8')`.

# Part 2: From One Element to All Entries: find_all, Numbers and Missing Values

The lesson page of this part: [From One Element to All Entries: find_all, Numbers and Missing Values](https://dvirross.com/courses/intro-data-science/data-collection-find-all)

In this lesson we learn how Beautiful Soup collects all the elements that fit, how a list comprehension turns them into data, how text that holds a number is converted to a number, and what to do when an element is missing from some of the entries.

## Which paper has the most authors?

Papers in hardware architecture are often the work of a team: some have one author and some have ten. On the arXiv list of October 2026 we ask how many authors a paper has, which paper has the most, and how many there are on average. The saved copy of the list holds the first 25 of its 266 entries, so the answers describe these 25 papers and not the whole month.

The first box requests the saved copy of the list and builds the soup, as in [From a Web Page to Data: Beautiful Soup](https://dvirross.com/courses/intro-data-science/data-collection-beautifulsoup), so that this page stands on its own.

In [22]:
import requests
from bs4 import BeautifulSoup

list_url = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/arxiv-cs-ar-2026-10.htm'
)
page_text = requests.get(list_url).text
soup = BeautifulSoup(page_text, 'html.parser')
print(soup.title.text)

Hardware Architecture Oct 2026


## `find_all`: every element that fits

Each paper of the list is a `dd` element that holds its title, its authors and its subjects. The method `find_all` returns all the elements that fit, as a list, in the order of the page. It takes the same arguments as `find`.

In [23]:
entries = soup.find_all('dd')
print(len(entries))

25


The list `entries` has 25 elements, one for each paper, and each one has its own `find`, which looks only inside it. The steps that read the title of one paper go into a function, `title_of`. A list comprehension builds a list from another list in one line: the expression before `for` is computed for every element, here the call of `title_of`.

In [24]:
def title_of(entry):
    box = entry.find('div', class_='list-title')
    return box.text.replace('Title:', '').strip()

titles = [title_of(entry) for entry in entries]
print(len(titles))
print(titles[0])
print(titles[1])

25
SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs
ShatterQuant: Breaking Uniform Precision with Block-Wise Mixed-Precision on a Systolic Transformer Hardware Accelerator


## Counting the authors

The authors of a paper are in a `div` of the class `list-authors`, and each author is a link, an element `a`. So the number of authors is the number of links inside that `div`: `find_all` returns a list and `len` counts it. Run the box and look at the result for the second paper, `entries[1]`: the number of links in its `list-authors`.

In [25]:
authors = entries[1].find('div', class_='list-authors')
print(len(authors.find_all('a')))

5


The steps that read one paper are the same for all papers, so we put them in a function, which returns the pair of the title and the number of authors, and call it on every entry. A pair like this is a tuple.

In [26]:
def read_entry(entry):
    authors = entry.find('div', class_='list-authors')
    return (title_of(entry), len(authors.find_all('a')))

papers = [read_entry(entry) for entry in entries]
print(papers[0])

('SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs', 3)


## The maximum and the mean

`papers` is a list of 25 pairs. The function `max` returns the largest item, and its argument `key` says what to compare: here a function that takes a pair and returns its second part, the number of authors, written as `lambda paper: paper[1]`. The mean is the sum divided by the count, and `round(number, 2)` keeps two digits after the decimal point.

In [27]:
top = max(papers, key=lambda paper: paper[1])
print(top[1], top[0])

counts = [paper[1] for paper in papers]
print(round(sum(counts) / len(counts), 2))

10 PEEK: Heterogeneous Parallelism for Privileged Error Detection in Safety-Critical Processors
4.2


Among these 25 papers the largest team has 10 authors, and the mean is 4.2 authors per paper. The mean is higher than a typical paper: half of the papers have 4 authors or fewer, and a few large teams raise the mean.

## Text that holds a number

The page says how many entries the month has, in the line `Total of 266 entries`, in a `div` of the class `paging`. The text is not a number, and a program cannot compute with it. The method `split()` cuts a text at its spaces and returns a list of its words; `[2]`, the third word, counted from 0, is `'266'`, and the function `int` converts the text of a whole number to a number.

In [28]:
paging = soup.find('div', class_='paging')
total = int(paging.text.split()[2])
print(total)
print(len(entries), 'of', total)

266
25 of 266


Without `int`, the program would hold the text `'266'`, and `'266' + 1` would be an error. A number written with a thousands separator, such as `3,330`, needs one more step, which is `replace(',', '')`, because `int` does not read the comma.

A number with a decimal point is converted with `float`, and a unit written after it is removed first. The page of a paper writes the size of its submission as `3,330 KB`, and the same steps give the number:

In [29]:
size_text = '3,330 KB'
number = size_text.replace(' KB', '').replace(',', '')
print(float(number))

3330.0


## An element that is missing

Not every paper has the line of comments, in which the authors write, for example, the number of pages. In the list, 13 of the 25 papers have no `div` of the class `list-comments` at all. For such an entry `find` returns `None`, and `None.text` is an error. A program that reads many entries tests for `None` before it reads.

In [30]:
without_comments = [
    entry for entry in entries
    if entry.find('div', class_='list-comments') is None
]
print(len(without_comments))

13


When the comments exist, they are free text written by the authors, such as `7 pages, 5 figures. Accepted to ASP-DAC 2027`. A program reads a number of pages only where the text says it, in the form that a number comes first and the word `pages` after it. The method `isdigit()` tells whether a text is made of digits only, and `startswith('pages')` whether a text begins with the word. The function returns `None` for every entry that does not say.

In [31]:
def pages_of(entry):
    box = entry.find('div', class_='list-comments')
    if box is None:
        return None
    words = box.text.replace('Comments:', '').split()
    if len(words) < 2:
        return None
    if words[0].isdigit() and words[1].startswith('pages'):
        return int(words[0])
    return None

In [32]:
all_pages = [pages_of(entry) for entry in entries]
print(all_pages)

[None, None, 7, None, 4, 4, None, None, None, None, 8, None, None, None, 10, 15, None, None, None, None, None, None, None, None, None]


A missing number is not zero. To compute with the numbers that exist, we keep only the items that are not `None`, and the mean is over those:

In [33]:
known = [pages for pages in all_pages if pages is not None]
print(len(known), max(known), sum(known) / len(known))

6 15 8.0


Only 6 of the 25 papers say how many pages they have, and these 6 have 8 pages on average and at most 15. The number describes the papers that say, which may be different from the papers that do not.

### Practice

**Your turn.** The list `papers` holds a pair of the title and the number of authors for each paper. Print the title of the paper that has the fewest authors; when several papers have the fewest, print the first of them in the list.

> **Hint**
>
> The function `min` works like `max`, with the same argument `key`. It returns the first of the items that have the smallest value.

**Solution**

In [34]:
fewest = min(papers, key=lambda paper: paper[1])
print(fewest[0])

A Compression Tree Generation Method for High Performance VLSI Datapaths


**Expected output**

```text
A Compression Tree Generation Method for High Performance VLSI Datapaths
```

In [35]:
check(4)

OK: the output is as expected.


**Common mistake**

```python
fewest = max(papers, key=lambda paper: paper[1])
print(fewest[0])
```

The function `max` returns the paper with the most authors, not the fewest.

**Your turn.** Print how many of the 25 papers have at least 5 authors, as one number.

> **Hint**
>
> Build a list of the pairs whose second part is at least 5 with a condition at the end of a list comprehension, and print its length.

**Solution**

In [36]:
large = [paper for paper in papers if paper[1] >= 5]
print(len(large))

9


**Expected output**

```text
9
```

In [37]:
check(5)

OK: the output is as expected.


**Common mistake**

```python
large = [paper for paper in papers if paper[1] > 5]
print(len(large))
```

The condition `> 5` leaves out the papers with exactly 5 authors, and "at least 5" includes them.

**Your turn.** The function `pages_of` above returns the number of pages of an entry, or `None`. Print the mean number of pages, rounded to one digit after the decimal point, over the papers that say how many pages they have.

> **Hint**
>
> Keep only the values that are not `None`, as in the list `known`, and divide their sum by their count, not by the number of entries.

**Solution**

In [38]:
numbers = [pages_of(entry) for entry in entries]
numbers = [n for n in numbers if n is not None]
print(round(sum(numbers) / len(numbers), 1))

8.0


**Expected output**

```text
8.0
```

In [39]:
check(6)

OK: the output is as expected.


**Common mistake**

```python
numbers = [pages_of(entry) for entry in entries]
numbers = [n for n in numbers if n is not None]
print(round(sum(numbers) / len(entries), 1))
```

The sum is over the 6 papers that say, but the division is by all 25 entries, which treats a missing number as if it were zero pages. The result, 1.9, describes no paper.

## Keep in mind

- `find_all` returns a list of all the elements that fit; an element of the list has its own `find` and `find_all`.
- A list comprehension turns the list of elements into a list of data; a function that reads one entry keeps the code in one place.
- The text of an element is text: `int(...)` converts text of a whole number, `float(...)` a number with a decimal point, `split()` cuts a text into words, and a unit or a thousands separator is removed with `replace` first.
- An element that is missing gives `None` from `find`; test `is None` before reading `text`, and keep a missing value out of a mean, because it is not zero.
- `max` and `min` with `key` return the item with the largest or the smallest value of a part of it.

# Part 3: From a List to Many Pages: Following Links

The lesson page of this part: [From a List to Many Pages: Following Links](https://dvirross.com/courses/intro-data-science/data-collection-linked-pages)

> **Self-study.** This page is for self-study: it is not taught in the lecture, and you read it on your own.

In this lesson we learn how a program follows the links of a list to the page of each item, reads data from the tags that describe a page, turns text such as `3,330 KB` into a number, and collects the results of many pages in one list.

## How large is the submission of each paper?

The list of papers shows the title, the authors and the subjects of each paper, but not the size of the files that the authors submitted. That number is on the page of each paper, which the list links to. The question is: among the first eight papers of the list, which one has the largest submission, and what is the median size? To answer it, a program has to read the list, follow eight links and read one number on each page.

The first box defines `soup_of`, a function that requests an address and returns the soup of its page, as in [From a Web Page to Data: Beautiful Soup](https://dvirross.com/courses/intro-data-science/data-collection-beautifulsoup), so that this page stands on its own. It builds the soup of the list and collects the links to the papers: every link whose attribute `title` is `Abstract`, as in [From One Element to All Entries](https://dvirross.com/courses/intro-data-science/data-collection-find-all). A slice, `[:8]`, keeps the first eight.

In [40]:
import requests
from bs4 import BeautifulSoup

copies = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/'
)

def soup_of(address):
    text = requests.get(address).text
    return BeautifulSoup(text, 'html.parser')

soup = soup_of(copies + 'arxiv-cs-ar-2026-10.htm')
anchors = soup.find_all('a', title='Abstract')
links = [a.get('href') for a in anchors][:8]
print(links[0])
print(len(links))

/abs/2610.00106
8


## From the link to the address of a page

The `href` of a paper is `/abs/2610.00106`: the identifier of the paper is its last part, and the address is relative to the site of the list, `https://arxiv.org`. On the live site the full address is the site name followed by the `href`. The saved copies of the papers are kept under names of the form `arxiv-abs-2610.00106.htm`, so a function turns an `href` into the address of its copy.

In [41]:
def copy_address(href):
    paper_id = href.replace('/abs/', '')
    return copies + 'arxiv-abs-' + paper_id + '.htm'

page = soup_of(copy_address(links[0]))
print(page.find('h1', class_='title').text)

Title:SyntheticHLS: Building Diverse Synthetic High-Level Synthesis Datasets using LLMs


## Data in the tags that describe the page

The head of every page holds `meta` elements: tags that are not drawn, which describe the page to search engines and to programs. Each has an attribute `name` that says what it holds, and an attribute `content` that holds the value, for example the date in which the paper was submitted. The first parameter of `find` is also called `name`, so writing `name='citation_date'` raises a `TypeError`; the attribute goes in the dictionary `attrs`, with the name that we want; `get('content')` reads its value.

In [42]:
date = page.find('meta', attrs={'name': 'citation_date'})
print(date.get('content'))

2026/09/09


The same page has one `meta` element for each author, with the name `citation_author`, so `find_all` with the same `attrs` counts the authors:

In [43]:
meta_name = {'name': 'citation_author'}
print(len(page.find_all('meta', attrs=meta_name)))

3


## A number inside text

The size of the submission is written in the section of the page called the submission history, in a `div` of the class `submission-history`, as part of a line of text: `[v1] Wed, 9 Sep 2026 02:40:29 UTC (3,330 KB)`. The number is between a bracket and the letters `KB`, and it has a comma. We take it in three steps. `split('(')` cuts the text at every opening bracket and returns a list of the pieces, and `[-1]`, the last piece, is `'3,330 KB)'` with a line break. These papers have one version each; with several versions the last piece is the size of the last one. `replace` removes the letters and the comma, `strip` removes the line break, and `int` converts the text to a number. The function `repr` shows a text as it is written, with its hidden characters, such as the line break `\n`.

In [44]:
box = page.find('div', class_='submission-history')
history = box.text
piece = history.split('(')[-1]
print(repr(piece))
number = piece.replace('KB)', '').replace(',', '')
kb = int(number.strip())
print(kb)

'3,330 KB)\n'
3330


Without the removal of the comma, `int('3,330')` raises an error.

## A missing element stops the program

Some papers have a line of comments in the table of the page, an element `td` of the class `comments`, and some do not. The comment of the first paper is there; the page of the second paper has none. Asking for the text of an element that is not there is an error:

On the page of the lesson the cell above stops with an error on purpose. In the notebook the cell below prints the element that `find` returned, so that the notebook runs to the end: it is `None`, and the error comes from asking `None` for `.text`.

In [45]:
second = soup_of(copy_address(links[1]))
print(second.find('td', class_='comments'))

`find` returned `None`, and `None` has no attribute `text`. A function that reads a page tests the element before it reads from it, and returns `None` for the value that does not exist. The function below reads one paper and returns its data as a dictionary: the identifier, the title from the `meta` tag, the size in KB, and the comment or `None`.

In [46]:
def read_paper(href):
    page = soup_of(copy_address(href))
    meta = {'name': 'citation_title'}
    title = page.find('meta', attrs=meta).get('content')
    box = page.find('div', class_='submission-history')
    piece = box.text.split('(')[-1]
    number = piece.replace('KB)', '').replace(',', '')
    kb = int(number.strip())
    comment = page.find('td', class_='comments')
    if comment is not None:
        comment = comment.text.strip()
    return {
        'id': href.replace('/abs/', ''),
        'title': title, 'kb': kb, 'comment': comment,
    }

second_paper = read_paper(links[1])
print(second_paper['kb'], second_paper['comment'])

19279 None


## All eight pages

A `for` loop, or a list comprehension, calls the function on each link. A program that sends many requests to one site waits between them, with `time.sleep(seconds)`, as [Reading pages responsibly](https://dvirross.com/courses/intro-data-science/data-collection-beautifulsoup#reading-pages-responsibly) explains; the copies on this site are not arXiv, so the box needs no pause.

In [47]:
papers = [read_paper(href) for href in links]
for paper in papers:
    print(paper['id'], paper['kb'])

2610.00106 3330
2610.00207 19279
2610.00311 856
2610.01186 866
2610.01603 492
2610.01623 234
2610.01867 747
2610.01918 159


The question of the lesson is answered by the maximum, the mean and the median of the sizes. The median is the middle of the sizes in order: `sorted` puts them in order, and with eight values it is the mean of the two middle ones, at places 3 and 4 (counted from 0).

In [48]:
sizes = [paper['kb'] for paper in papers]
largest = max(papers, key=lambda paper: paper['kb'])
print(largest['kb'], largest['title'][:40])
print(round(sum(sizes) / len(sizes)))
ordered = sorted(sizes)
print((ordered[3] + ordered[4]) / 2)

19279 ShatterQuant: Breaking Uniform Precision
3245
801.5


The largest submission is 19,279 KB, the mean is 3,245 KB and the median is 801.5 KB. One large paper pulls the mean up (the other seven have a mean of 955 KB) but not the median, so the median describes a typical paper.

### Practice

**Your turn.** The function `read_paper` and the list `papers` are above. Print how many of the eight papers have no comment on their page.

> **Hint**
>
> In `papers`, a paper without a comment has `None` in its key `'comment'`. Count with a list comprehension and a condition `is None`.

**Solution**

In [49]:
without = [p for p in papers if p['comment'] is None]
print(len(without))

3


**Expected output**

```text
3
```

In [50]:
check(7)

OK: the output is as expected.


**Common mistake**

```python
without = [p for p in papers if p['comment'] == '']
print(len(without))
```

A missing comment is `None`, and `None` is not the empty text `''`, so the condition is never true and the program prints 0.

**Your turn.** The saved copy of the page of the paper `2610.00311` is at `copies + 'arxiv-abs-2610.00311.htm'`. Print the date of its submission, from the `meta` tag `citation_date`.

> **Hint**
>
> Build the soup of the page, find the `meta` element with `attrs={'name': 'citation_date'}` (not `name='citation_date'`, which raises a `TypeError`) and read its attribute `content` with `get`.

**Solution**

In [51]:
other = soup_of(copies + 'arxiv-abs-2610.00311.htm')
date = other.find('meta', attrs={'name': 'citation_date'})
print(date.get('content'))

2026/09/28


**Expected output**

```text
2026/09/28
```

In [52]:
check(8)

OK: the output is as expected.


The attribute `name` is written in `attrs`: writing `name='citation_date'` would raise a `TypeError`, because `name` is the first parameter of `find`.

**Common mistake**

```python
other = soup_of(copies + 'arxiv-abs-2610.00311.htm')
date = other.find('meta', attrs={'name': 'citation_date'})
print(date)
```

The program prints the whole `meta` tag. The date is the value of its attribute `content`, which `get('content')` reads.

**Your turn.** Print the mean size in KB of the seven papers other than the largest one, rounded to a whole number.

> **Hint**
>
> Take the sizes in `sizes`, remove the largest with `max(sizes)` by a condition in a list comprehension, and divide the sum of the others by their count.

**Solution**

In [53]:
others = [kb for kb in sizes if kb != max(sizes)]
print(round(sum(others) / len(others)))

955


**Expected output**

```text
955
```

In [54]:
check(9)

OK: the output is as expected.


**Common mistake**

```python
others = [kb for kb in sizes if kb != max(sizes)]
print(round(sum(others) / len(sizes)))
```

The sum is over seven papers, and the division is by eight, the length of the full list, so the mean is too small.

**Live site.** The cell below can read the pages of the first three papers of the list from arXiv itself, with a user agent that says how to reach us and the pause that `robots.txt` asks for, 15 seconds between requests; it takes about a minute. It is switched off (`RUN_LIVE = False`): a whole class that runs it together sends many requests from one address. One person, once, can set `RUN_LIVE = True`. If the site answers 403 or 429, the cell stops and says so; it does not try again. The rest of the notebook works on the saved copies.

In [55]:
import time
RUN_LIVE = False  # switch to True to ask arXiv itself (four requests, 15 seconds apart)
if RUN_LIVE:
    live_headers = {'User-Agent': 'IntroDataScienceCourse/1.0 (teaching example; contact via dvirross.com)'}
    live_list = requests.get('https://arxiv.org/list/cs.AR/2026-10?skip=0&show=25', headers=live_headers, timeout=30)
    refused = live_list.status_code in (403, 429)
    if not refused:
        live_soup = BeautifulSoup(live_list.text, 'html.parser')
        for link in live_soup.find_all('a', title='Abstract')[:3]:
            if not refused:
                time.sleep(15)
                live_page = requests.get('https://arxiv.org' + link.get('href'), headers=live_headers, timeout=30)
                print(live_page.status_code, link.get('href'))
                refused = live_page.status_code in (403, 429)
    if refused:
        print('The site refused a request or asked to slow down (status 403 or 429). Do not try again now; wait and use the saved copies.')
else:
    print('The live cell is off. Set RUN_LIVE = True to run it.')

## Keep in mind

- A list page links to the page of each item; the `href` of a link is the way, and a function that turns an `href` into an address keeps the code in one place.
- `find('meta', attrs={'name': ...})` and `get('content')` read the data that a page keeps in its `meta` tags.
- A number inside text is cut out with `split`, cleaned with `replace` and `strip`, and converted with `int`.
- An element that is missing is `None`; asking it for `text` raises an error, so a function tests for `None` and returns `None`.
- The mean of a few values is pulled by a very large value; look at the maximum and the median next to the mean.
- A program that reads a site keeps to its `robots.txt`, waits between requests, says who it is and takes only the data it needs.